# Figure 10 - Effective static structure factor S(q), full potential

Many-body Monte Carlo simulations with **Faunus** (N = 20 rigid antibodies at
amino-acid resolution, Yukawa + Lennard-Jones) give the measured scattering
I(q). A separate single-molecule run (N = 1, same box size) gives the form
factor P(q). The effective structure factor is their ratio, S(q) = I(q)/P(q).

Two run blocks are therefore needed:
* **P_q/** - form factor P(q), one molecule, one replica
* **I_q/** - many-body I(q), N = 20, three replicas

Runs cover `mAb-1` (7 and 57 mM structures) at 20, 50, 150 mg/mL, reading
structures from the shared `../pqr_inputs/` folder. The plot assembles the 2x2
S(q) panel (saved as `S_q_2x2.pdf`).

**How to run:** install the environment, then run top to bottom (P_q before I_q
before the plot). Edit `run_faunus()` to submit to a cluster instead. The
sample `micro` values give short runs; increase them for production statistics.

## Import packages

In [1]:
import os
import subprocess
import numpy as np
from scipy import interpolate
from jinja2 import Template
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 16, "legend.frameon": True, "figure.figsize": [12, 8],
                     "xtick.major.size": 7, "ytick.major.size": 7, "legend.labelspacing": 1})


## Functions

In [2]:
def Length_in_gl(cp, N_of_proteins, Mw):
    """Cubic box side (angstrom) giving protein concentration cp (mg/mL)
    for N_of_proteins molecules of molar mass Mw (g/mol)."""
    return np.cbrt(N_of_proteins / (cp * (1 / Mw) * 6.022e23 * 1e-27))


def create_input(workdir, structure, macro, micro, nstep, nskip, L, N, Dl, eps):
    """Render the Faunus input (input.yml). L box side (angstrom), N antibodies,
    Dl Debye length (angstrom), eps Lennard-Jones well depth (kJ/mol)."""
    with open("input.yml", "w") as f:
        return f.write(Input.render(workdir=workdir, structure=structure,
                                    macro=macro, micro=micro, nstep=nstep,
                                    nskip=nskip, L=L, N=N, Dl=Dl, eps=eps))


def run_faunus(use_state=False):
    """Run Faunus locally in the current run directory.

    Requires `faunus` and `yason.py` on PATH (installed by the conda
    environment, see ../environment.yml). Output goes to `out`, errors to `err`.
    To run on a cluster instead, replace the body of this function with your
    own submission command."""
    state = "--state state.json " if use_state else ""
    cmd = ("set -o pipefail; export OMP_NUM_THREADS=1; "
           f"yason.py input.yml | faunus {state}-v 6 > out 2> err")
    subprocess.run(["bash", "-c", cmd], check=True)


def kappa(conc, Z, rhob, rhos, sigma):
    """Inverse Debye length (1/nm). conc is 0 in these runs, so the screening
    depends on the salt term rhob (ionic strength, mM) only."""
    lB = 0.714
    Na = 6.02214076e23
    rho = 4.068e15
    phi = conc * rho * 1e-21 * np.pi * sigma ** 3 / 6
    PHI = 1 / (1 - phi)
    k = 4 * np.pi * lB * (PHI * conc * Z * rho * 1e-21
                          + 2 * rhob * Na * 1e-27 + 2 * rhos * Na * 1e-27)
    return np.sqrt(k)


# Form factor P(q) - single-molecule runs

In [4]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------

Input = Template("""
temperature: 298.15
geometry: {type: cuboid, length: [{{L}},{{L}},{{L}}]}
mcloop: {micro: {{micro}}, macro: {{macro}}}
random: {seed: hardware}
energy:
    - nonbonded_coulomblj:
        cutoff_g2g: {{ 180 + 8*Dl }}
        coulomb: { type: yukawa,  debyelength: {{Dl}}, epsr: 78.7, shift: True,  cutoff: {{8*Dl}} }
        lennardjones: { mixing: LB}

atomlist:
    - CTR:  {mw: 16.0,  sigma: 3.13, eps: {{eps}}, q: -1 }
    - HCTR: {mw: 16.0,  sigma: 3.13, eps: {{eps}}, q: 0 }
    - NTR:  {mw: 14.0,  sigma: 2.99, eps: {{eps}}, q: 0 }
    - HNTR: {mw: 14.0,  sigma: 2.99, eps: {{eps}}, q: 1 }
    - ALA:  {mw: 66.0,  sigma: 5.02, eps: {{eps}}, q: 0 }
    - ILE:  {mw: 102.0, sigma: 5.8,  eps: {{eps}}, q: 0 }
    - LEU:  {mw: 102.0, sigma: 5.8,  eps: {{eps}}, q: 0 }
    - MET:  {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 0 }
    - PHE:  {mw: 138.0, sigma: 6.41, eps: {{eps}}, q: 0 }
    - VAL:  {mw: 90.0,  sigma: 5.56, eps: {{eps}}, q: 0 }
    - PRO:  {mw: 90.0,  sigma: 5.56, eps: {{eps}}, q: 0 }
    - TRP:  {mw: 176.0, sigma: 6.95, eps: {{eps}}, q: 0 }
    - GLY:  {mw: 54.0,  sigma: 4.69, eps: {{eps}}, q: 0 }
    - ARG:  {mw: 144.0, sigma: 6.50, eps: {{eps}}, q: 0 }
    - HARG: {mw: 144.0, sigma: 6.50, eps: {{eps}}, q: 1 }
    - LYS:  {mw: 116.0, sigma: 6.05, eps: {{eps}}, q: 0 }
    - HLYS: {mw: 116.0, sigma: 6.05, eps: {{eps}}, q: 1 }
    - ASP:  {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: -1 }
    - HASP: {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: 0 }
    - LASP: {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: 2 }
    - GLU:  {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: -1 }
    - HGLU: {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 0 }
    - LGLU: {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 2 }
    - GLN:  {mw: 120.0, sigma: 6.12, eps: {{eps}}, q: 0 }
    - ASN:  {mw: 108.0, sigma: 5.91, eps: {{eps}}, q: 0 }
    - HIS:  {mw: 130.0, sigma: 6.29, eps: {{eps}}, q: 0 }
    - HHIS: {mw: 130.0, sigma: 6.29, eps: {{eps}}, q: 1 }
    - SER:  {mw: 82.0,  sigma: 5.39, eps: {{eps}}, q: 0 }
    - THR:  {mw: 94.0,  sigma: 5.64, eps: {{eps}}, q: 0 }
    - TYR:  {mw: 154.0, sigma: 6.65, eps: {{eps}}, q: -1 }
    - HTYR: {mw: 154.0, sigma: 6.65, eps: {{eps}}, q: 0 }
    - CYS:  {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: -1 }
    - HCYS: {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: 0 }
    - CYX:  {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: 0 }                

moleculelist:
    - mAb_aa:
        structure: {{workdir}}/../pqr_inputs/{{structure}}
        keepcharges: True
        keeppos: False
        rigid: True

insertmolecules:
    - mAb_aa: {N: 1, inactive: False}
moves:    
    - moltransrot: {molecule: mAb_aa, dir: [1,1,1], dprot: 0.2, dp: 100 }

analysis:
    - sanity: {nstep: {{nstep}} }
    - molecule_density: {nstep: {{nstep}}}
    - savestate: {file: state.json}
    - savestate: {file: confout.pqr}
    - xtcfile: {file: traj.xtc, nstep: {{nstep}}, nskip: {{nskip}} }
    - scatter: {file: debye.dat, nstep: 1, molecules: ["mAb_aa"], com: False, scheme: explicit, pmax: 15 }
    - scatter: {file: debye_COM.dat, nstep: 1, molecules: ["mAb_aa"], com: True, scheme: explicit, pmax: 15 } 

""")

Mw = 148000                              # antibody molar mass, g/mol
cp_range = [20, 50, 150]                 # protein concentrations, mg/mL
to_kJmol = 2.476
eps = round(0.075 * to_kJmol, 4)         # Lennard-Jones well depth, kJ/mol
N = 20                                   # box is sized for 20 antibodies
structures_list = ['mAb-1_7mM.pqr', 'mAb-1_57mM.pqr']
replicas = [0]

micro = 100
macro = 1000
nstep = 1000
nskip = 0

os.makedirs('P_q', exist_ok=True)
os.chdir('P_q')
base = os.getcwd()

for structure in structures_list:
    name = os.path.splitext(structure)[0]

    if structure == 'mAb-1_7mM.pqr':
        z, sigma, I = 31.3, 9.64, 7
    elif structure == 'mAb-1_57mM.pqr':
        z, sigma, I = 36.3, 9.0, 57
    Dl = 10 / kappa(0, z, I, 0, sigma)   # Debye length, angstrom

    for cp in cp_range:
        L = Length_in_gl(cp, N, Mw)      # box side for this concentration
        for r in replicas:
            rundir = f"{name}_cp{cp}_r{r}"
            os.makedirs(rundir, exist_ok=True)
            os.chdir(rundir)
            create_input(workdir, structure, macro, micro, nstep, nskip, L, N, Dl, eps)
            run_faunus(use_state=os.path.isfile('state.json'))
            os.chdir(base)
os.chdir(workdir)

KeyboardInterrupt: 

# Structure factor I(q) - many-body runs

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------

Input = Template("""
temperature: 298.15
geometry: {type: cuboid, length: [{{L}},{{L}},{{L}}]}
mcloop: {micro: {{micro}}, macro: {{macro}}}
random: {seed: hardware}
energy:
    - nonbonded_coulomblj:
        cutoff_g2g: {{ 180 + 8*Dl }}
        coulomb: { type: yukawa,  debyelength: {{Dl}}, epsr: 78.7, shift: True,  cutoff: {{8*Dl}} }
        lennardjones: { mixing: LB}

atomlist:
    - CTR:  {mw: 16.0,  sigma: 3.13, eps: {{eps}}, q: -1 }
    - HCTR: {mw: 16.0,  sigma: 3.13, eps: {{eps}}, q: 0 }
    - NTR:  {mw: 14.0,  sigma: 2.99, eps: {{eps}}, q: 0 }
    - HNTR: {mw: 14.0,  sigma: 2.99, eps: {{eps}}, q: 1 }
    - ALA:  {mw: 66.0,  sigma: 5.02, eps: {{eps}}, q: 0 }
    - ILE:  {mw: 102.0, sigma: 5.8,  eps: {{eps}}, q: 0 }
    - LEU:  {mw: 102.0, sigma: 5.8,  eps: {{eps}}, q: 0 }
    - MET:  {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 0 }
    - PHE:  {mw: 138.0, sigma: 6.41, eps: {{eps}}, q: 0 }
    - VAL:  {mw: 90.0,  sigma: 5.56, eps: {{eps}}, q: 0 }
    - PRO:  {mw: 90.0,  sigma: 5.56, eps: {{eps}}, q: 0 }
    - TRP:  {mw: 176.0, sigma: 6.95, eps: {{eps}}, q: 0 }
    - GLY:  {mw: 54.0,  sigma: 4.69, eps: {{eps}}, q: 0 }
    - ARG:  {mw: 144.0, sigma: 6.50, eps: {{eps}}, q: 0 }
    - HARG: {mw: 144.0, sigma: 6.50, eps: {{eps}}, q: 1 }
    - LYS:  {mw: 116.0, sigma: 6.05, eps: {{eps}}, q: 0 }
    - HLYS: {mw: 116.0, sigma: 6.05, eps: {{eps}}, q: 1 }
    - ASP:  {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: -1 }
    - HASP: {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: 0 }
    - LASP: {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: 2 }
    - GLU:  {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: -1 }
    - HGLU: {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 0 }
    - LGLU: {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 2 }
    - GLN:  {mw: 120.0, sigma: 6.12, eps: {{eps}}, q: 0 }
    - ASN:  {mw: 108.0, sigma: 5.91, eps: {{eps}}, q: 0 }
    - HIS:  {mw: 130.0, sigma: 6.29, eps: {{eps}}, q: 0 }
    - HHIS: {mw: 130.0, sigma: 6.29, eps: {{eps}}, q: 1 }
    - SER:  {mw: 82.0,  sigma: 5.39, eps: {{eps}}, q: 0 }
    - THR:  {mw: 94.0,  sigma: 5.64, eps: {{eps}}, q: 0 }
    - TYR:  {mw: 154.0, sigma: 6.65, eps: {{eps}}, q: -1 }
    - HTYR: {mw: 154.0, sigma: 6.65, eps: {{eps}}, q: 0 }
    - CYS:  {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: -1 }
    - HCYS: {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: 0 }
    - CYX:  {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: 0 }                

moleculelist:
    - mAb_aa:
        structure: {{workdir}}/../pqr_inputs/{{structure}}
        keepcharges: True
        keeppos: False
        rigid: True

insertmolecules:
    - mAb_aa: {N: {{N}}, inactive: False}
moves:    
    - moltransrot: {molecule: mAb_aa, dir: [1,1,1], dprot: 0.2, dp: 100 }

analysis:
    - sanity: {nstep: {{nstep}} }
    - molecule_density: {nstep: {{nstep}}}
    - savestate: {file: state.json}
    - savestate: {file: confout.pqr}
    - xtcfile: {file: traj.xtc, nstep: {{nstep}}, nskip: {{nskip}} }
    - scatter: {file: debye.dat, nstep: 1, molecules: ["mAb_aa"], com: False, scheme: explicit, pmax: 15 }
    - scatter: {file: debye_COM.dat, nstep: 1, molecules: ["mAb_aa"], com: True, scheme: explicit, pmax: 15 } 

""")

Mw = 148000                              # antibody molar mass, g/mol
cp_range = [20, 50, 150]                 # protein concentrations, mg/mL
to_kJmol = 2.476
eps = round(0.075 * to_kJmol, 4)         # Lennard-Jones well depth, kJ/mol
N = 20                                   # box is sized for 20 antibodies
structures_list = ['mAb-1_7mM.pqr', 'mAb-1_57mM.pqr']
replicas = [0, 1, 2]

micro = 10
macro = 1000
nstep = 1000
nskip = 0

os.makedirs('I_q', exist_ok=True)
os.chdir('I_q')
base = os.getcwd()

for structure in structures_list:
    name = os.path.splitext(structure)[0]

    if structure == 'mAb-1_7mM.pqr':
        z, sigma, I = 31.3, 9.64, 7
    elif structure == 'mAb-1_57mM.pqr':
        z, sigma, I = 36.3, 9.0, 57
    Dl = 10 / kappa(0, z, I, 0, sigma)   # Debye length, angstrom

    for cp in cp_range:
        L = Length_in_gl(cp, N, Mw)      # box side for this concentration
        for r in replicas:
            rundir = f"{name}_cp{cp}_r{r}"
            os.makedirs(rundir, exist_ok=True)
            os.chdir(rundir)
            create_input(workdir, structure, macro, micro, nstep, nskip, L, N, Dl, eps)
            run_faunus(use_state=os.path.isfile('state.json'))
            os.chdir(base)
os.chdir(workdir)


# Plot $S(q)=I(q)/P(q)$ for mAb-1 (7 and 57 mM) - Figure 10

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------


### Figure style ###
plt.rcParams.update({'font.size':16,'legend.frameon':True,'figure.figsize':[11,9],
                     'xtick.major.size':7,'ytick.major.size':7,'legend.labelspacing':1})

replicas = [0, 1, 2]                  # I_q replicas (P_q has _r0 only)
q_extrap = 0.01                       # extrapolate down to this q [nm^-1]

# --- Panel definitions: label, structure, and one or more series ---
panels = [
    dict(label='A', structure='mAb-1_7mM.pqr',  series=[(20,  'o', 'tab:blue', False),
                                                        (50,  's', 'tab:red',  False)]),
    dict(label='B', structure='mAb-1_57mM.pqr',  series=[(20,  '^', 'tab:blue', True)]),
    dict(label='C', structure='mAb-1_7mM.pqr',  series=[(150, 'o', 'tab:blue', True)]),
    dict(label='D', structure='mAb-1_57mM.pqr',  series=[(150, 'o', 'tab:blue', True)]),
]
row_titles = {'mAb-1_7mM.pqr':  'I = 7 mM',
              'mAb-1_57mM.pqr': 'I = 57 mM'}
marker_names = {'o': 'circles', 's': 'squares', '^': 'triangles'}
#######################################################################################################################################################


def load_S(workdir, name, cp):
    '''Replica-averaged S(q) = I_q/P_q for one (structure, concentration).'''
    P_dir = os.path.join(workdir, 'P_q', f"{name}_cp{cp}_r0")
    q_P, P_q = np.loadtxt(os.path.join(P_dir, 'debye.dat'),
                          unpack=True, skiprows=1, usecols=(0,1))

    S_reps = []
    for r in replicas:
        I_dir = os.path.join(workdir, 'I_q', f"{name}_cp{cp}_r{r}")
        q, I_r = np.loadtxt(os.path.join(I_dir, 'debye.dat'),
                            unpack=True, skiprows=1, usecols=(0,1))
        S_reps.append(I_r / np.interp(q, q_P, P_q))

    q_nm = q * 10                      # 1/Å -> 1/nm
    return q_nm, np.array(S_reps), np.mean(S_reps, axis=0)

fig, ax = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True)

for p, pan in enumerate(panels):
    r, c = divmod(p, 2)
    a = ax[r, c]
    name = os.path.splitext(pan['structure'])[0]

    for (cp, marker, color, filled) in pan['series']:
        q_nm, S_reps, S_avg = load_S(workdir, name, cp)

        # single replicas (faint)
        for S_r in S_reps:
            a.plot(q_nm, S_r, marker,
                   color=color, markerfacecolor='none',
                   markersize=3, alpha=0.25)

        # average: open or solid markers only (no line)
        a.plot(q_nm, S_avg, marker,
               color=color,
               markerfacecolor=color if filled else 'none',
               markeredgecolor=color, markersize=7,
               label=f'$c_p$ = {cp} mg/mL ({marker_names[marker]})')

        # extrapolation: from the smallest simulated q down to q_extrap
        S_inter = interpolate.interp1d(q_nm, S_avg, fill_value='extrapolate')
        q_low = np.arange(q_extrap, q_nm.min(), 0.05)
        a.plot(q_low, S_inter(q_low),  marker,
               color='black', markerfacecolor='black', markersize=7, alpha=0.3)


    a.set_xscale('log')
    a.set_xlim(0.05, 3.0)
    a.set_ylim(0, 1.5)
    a.set_xticks([0.1, 1])
    a.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    a.minorticks_off()

    a.set_ylabel("S(q)", fontsize=25)
    a.set_title(f"{pan['label']}  ({row_titles[pan['structure']]})",
                loc='left', fontsize=13)
    a.legend(fontsize=9, frameon=True, loc='lower right')

# x label only under the bottom row
for c in range(2):
    ax[-1, c].set_xlabel("q [nm$^{-1}$]", fontsize=25)

plt.tight_layout()
plt.savefig(os.path.join(workdir, 'S_q_2x2.pdf'), bbox_inches='tight')
plt.show()
